# Vamana runs

End-to-end pipeline for one dataset:

1. **Convert** an ANN-benchmarks `.hdf5` into the `.fbin` format ParlayANN reads.
2. **Run Vamana** to build the graph (shells out to ParlayANN's `neighbors` binary).
3. **Compute coverage stats** and write the `(neighbor, uncov)` adj-list.
4. **Run modified Vamana**, whose pruning stops on achieved coverage (`-gamma`)
   rather than a fixed max out-degree, and compute its coverage the same way.
5. **Compare** the two graphs on coverage, out-degree, edge count and build time.

Step 3 uses the GPU when CuPy is available and falls back to NumPy otherwise.
Sections 1-3 use ParlayANN's stock binary; section 4 uses this project's copy in
`code/vamana/`, so both must be built before running the whole notebook.

## Configuration

Everything the run needs is set here; the rest of the notebook reads these.

In [20]:
from pathlib import Path

import numpy as np

from tqdm import tqdm

# --- paths ---------------------------------------------------------------
BASE_PATH   = "/scratch/pa2439/ANN-Search/datasets"
HDF5_PATH   = Path(f"{BASE_PATH}/coco_i2i-512-angular.hdf5")
DATA_DIR    = Path(f"{BASE_PATH}/coco_i2i-512-angular")                         # fbin output
BUILT_DIR   = Path("/scratch/pa2439/ANN-Search/navigable_graph_results/new_results")  # graph + adj-list
VAMANA_BIN  = Path("../ParlayANN/algorithms/vamana/neighbors")
MOD_VAMANA_BIN = Path("vamana/neighbors")   # this project's copy, with -gamma

# --- vamana build params -------------------------------------------------
R     = 32   # max out-degree
L     = 64   # beam width during construction (needs L >= R)
ALPHA = 1.0   # prune slack; 1.0 = strict pruning

# --- modified vamana (gamma stopping rule) -------------------------------
GAMMA  = 0.9    # stop once edges alpha-cover this fraction of the sample
SAMPLE = None   # -S sample size; None uses the default S = ceil(10 ln n)

# --- coverage ------------------------------------------------------------
COVERAGE_ALPHA = 1.0    # alpha the coverage is measured at
LIMIT = None            # cap nodes for a quick check; None = whole graph
DTYPE = "float64"       # float32 halves memory but makes boundary ties unstable

DATASET = "coco_i2i-512-euclidean"

TAG = f"vamana-{DATASET}-R{R}-alpha{ALPHA:g}"
GRAPH_PATH = BUILT_DIR / TAG
ADJ_PATH   = BUILT_DIR / f"adj-list-{TAG}.txt"

# Modified run writes to its own tag so the two never overwrite each other.
MOD_TAG        = f"modvamana-{DATASET}-R{R}-alpha{ALPHA:g}-gamma{GAMMA:g}"
MOD_GRAPH_PATH = BUILT_DIR / MOD_TAG
MOD_ADJ_PATH   = BUILT_DIR / f"adj-list-{MOD_TAG}.txt"
BASE_FBIN  = DATA_DIR / "base.fbin"
QUERY_FBIN = DATA_DIR / "query.fbin"

for d in (DATA_DIR, BUILT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"graph        -> {GRAPH_PATH}")
print(f"adj-list     -> {ADJ_PATH}")
print(f"mod graph    -> {MOD_GRAPH_PATH}")
print(f"mod adj-list -> {MOD_ADJ_PATH}")

graph        -> /scratch/pa2439/ANN-Search/navigable_graph_results/new_results/vamana-coco_i2i-512-euclidean-R32-alpha1
adj-list     -> /scratch/pa2439/ANN-Search/navigable_graph_results/new_results/adj-list-vamana-coco_i2i-512-euclidean-R32-alpha1.txt
mod graph    -> /scratch/pa2439/ANN-Search/navigable_graph_results/new_results/modvamana-coco_i2i-512-euclidean-R32-alpha1-gamma0.9
mod adj-list -> /scratch/pa2439/ANN-Search/navigable_graph_results/new_results/adj-list-modvamana-coco_i2i-512-euclidean-R32-alpha1-gamma0.9.txt


## 1. HDF5 to fbin

An `.fbin` file is a little-endian int32 `n`, an int32 `dims`, then `n * dims` row-major
float32 values. The `train` and `test` datasets are written to separate files.

In [21]:
import h5py


def write_fbin(vectors, path):
    """Write a 2D array to `path` in fbin format."""
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    n, dims = vectors.shape
    with open(path, "wb") as f:
        np.array([n, dims], dtype=np.int32).tofile(f)
        vectors.tofile(f)
    print(f"wrote {path} ({n} points, dimension {dims})")


def read_fbin(path):
    """Read an fbin file back into a 2D array."""
    with open(path, "rb") as f:
        n, dims = np.fromfile(f, dtype=np.int32, count=2)
        return np.fromfile(f, dtype=np.float32).reshape(n, dims)


def hdf5_to_fbin(hdf5_path, base_out, query_out):
    """Convert the train and test datasets into separate fbin files."""
    with h5py.File(hdf5_path, "r") as f:
        write_fbin(f["train"][:], base_out)
        write_fbin(f["test"][:], query_out)

In [22]:
if BASE_FBIN.exists() and QUERY_FBIN.exists():
    print(f"fbin files already present in {DATA_DIR}, skipping conversion")
else:
    hdf5_to_fbin(HDF5_PATH, BASE_FBIN, QUERY_FBIN)

fbin files already present in /scratch/pa2439/ANN-Search/datasets/coco_i2i-512-angular, skipping conversion


## 2. Run Vamana

Shells out to ParlayANN's `neighbors`. Build the binary first with `make` in
`ParlayANN/algorithms/vamana/`. Output is streamed so the pass progress is visible.

In [23]:
import re
import subprocess
import time

# "Graph built in 2.888 seconds" - the binary's own build timer, which excludes
# reading the fbin and writing the graph. Falls back to wall-clock if absent.
BUILD_TIME_RE = re.compile(r"Graph built in ([0-9.]+) seconds")


def run_vamana(binary, base_path, graph_out, R, L, alpha, extra_args=(),
               gamma=None, sample_size=None):
    """Build a Vamana index, streaming the binary's output.

    Returns (graph_out, build_seconds). Pass `gamma` to use the modified
    binary's coverage stopping rule; `sample_size` maps to -S and may be None.
    """
    binary = Path(binary).resolve()
    if not binary.exists():
        raise FileNotFoundError(
            f"{binary} not found - run `make` in its directory first")

    cmd = [
        str(binary),
        "-R", str(R), "-L", str(L), "-alpha", str(alpha),
        "-data_type", "float", "-dist_func", "Euclidian",
        "-base_path", str(Path(base_path).resolve()),
        "-graph_outfile", str(Path(graph_out).resolve()),
    ]
    if gamma is not None:
        cmd += ["-gamma", str(gamma)]
        if sample_size is not None:
            cmd += ["-S", str(sample_size)]
    cmd += list(extra_args)
    print(" ".join(cmd), "\n")

    started = time.perf_counter()
    build_seconds = None
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
        m = BUILD_TIME_RE.search(line)
        if m:
            build_seconds = float(m.group(1))
    if proc.wait() != 0:
        raise RuntimeError(f"neighbors exited with status {proc.returncode}")
    if build_seconds is None:                    # binary changed its output
        build_seconds = time.perf_counter() - started
    return graph_out, build_seconds


In [24]:
_, VAMANA_BUILD_SECONDS = run_vamana(
    VAMANA_BIN, BASE_FBIN, GRAPH_PATH, R, L, ALPHA, extra_args=["-verbose"])
print(f"\nbuild time: {VAMANA_BUILD_SECONDS:.3f}s")


/scratch/pa2439/ANN-Search/navigable_graph_results/ParlayANN/algorithms/vamana/neighbors -R 32 -L 64 -alpha 1.0 -data_type float -dist_func Euclidian -base_path /scratch/pa2439/ANN-Search/datasets/coco_i2i-512-angular/base.fbin -graph_outfile /scratch/pa2439/ANN-Search/navigable_graph_results/new_results/vamana-coco_i2i-512-euclidean-R32-alpha1 -verbose 

Data: detected 113287 points with dimension 512
Building graph...
number of passes = 1
Pass 10% complete
Pass 20% complete
Pass 30% complete
Pass 40% complete
Pass 50% complete
Pass 60% complete
Pass 70% complete
Pass 80% complete
Pass 90% complete
Pass 100% complete
beam search time: total: 8.3667
bidirect time: total: 1.1598
prune time: total: 1.6883
Average visited: 72, Tail visited: 88
Vamana graph built with 113287 points and parameters R = 32, L = 64
Graph has average degree 14.75 and maximum degree 32
Graph built in 14.87 seconds
Parlay time: 15.1170
Writing graph with 113287 points and max degree 32 to /scratch/pa2439/ANN-Sear

## 3. Coverage stats

ParlayANN's graph file stores only edges, so the uncovered counts are recomputed with the
same alpha-reachability rule as `distributed_robust_prune.py`: waypoint `w` covers `p`
when `d(w, p) * alpha < d(s, p)`. Distances are squared euclidean, so the test scales by
`alpha^2` and `p` stays uncovered while `d2(s, p) <= d2(w, p) * alpha_sq`.

Each edge records the uncovered count *after* it takes effect, matching the adj-list
writer. Edges are replayed in the order ParlayANN stored them.

Distances are accumulated in `DTYPE` (float64 by default). The covering test is an
inequality, so points sitting exactly on the `d(s, p) == d(w, p)` boundary - common at
`alpha = 1` - can flip between a blocked matmul and a per-vector pass when rounding in
float32. float64 removes that ambiguity and keeps counts reproducible.

The write is resumable: rerunning with the same `out_path` scans the lines already
there, drops a trailing line left half-written by an interrupt, and continues from
the first node still missing. Pass `resume=False` to start over from scratch.


In [67]:
def read_csr(path):
    """Read a ParlayANN graph file into (list of neighbor arrays, max_deg)."""
    with open(path, "rb") as f:
        n, max_deg = np.fromfile(f, dtype=np.uint32, count=2)
        sizes = np.fromfile(f, dtype=np.uint32, count=int(n))
        edges = np.fromfile(f, dtype=np.uint32)
    offsets = np.concatenate([[0], np.cumsum(sizes, dtype=np.int64)])
    return [edges[offsets[i]:offsets[i + 1]] for i in range(int(n))], int(max_deg)

In [68]:
# Use the GPU when CuPy is importable and a device is actually present.
try:
    import cupy as cp
    cp.cuda.runtime.getDeviceCount()
    xp = cp
    print("using GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
except Exception as exc:
    xp = np
    print(f"using CPU ({type(exc).__name__}: {exc})")

on_gpu = xp is not np

using GPU: NVIDIA L40S


In [69]:
def neighborhood_with_uncov(source, neighbors, V, norms, alpha_sq, chunk=32):
    """Replay one node's edges, recording the uncovered count after each edge.

    Distances to a block of waypoints come from one matmul, and the block is
    restricted to the rows still uncovered - that set shrinks quickly, so later
    blocks do far less work than a full pass over all points.
    """
    p = V[source]
    d_source = norms - 2.0 * (V @ p) + p @ p
    uncov = xp.arange(len(V), dtype=xp.int32)
    uncov = uncov[uncov != source]
    d_uncov = d_source[uncov]

    neighbors = xp.asarray(np.asarray(neighbors, dtype=np.int64))
    out = []
    for start in range(0, len(neighbors), chunk):
        block = neighbors[start:start + chunk]
        if len(uncov) == 0:                       # nothing left to cover
            out.extend((int(w), 0) for w in block.tolist())
            continue

        # (|uncov|, dim) @ (dim, block) -> squared distances for the whole block
        D = (norms[uncov][:, None]
             - 2.0 * (V[uncov] @ V[block].T)
             + norms[block][None, :])

        for t, w in enumerate(block.tolist()):
            keep = d_uncov <= D[:, t] * alpha_sq
            uncov, d_uncov, D = uncov[keep], d_uncov[keep], D[keep]
            # Drop the waypoint itself: for duplicate points d(w, w) = 0 makes the
            # test true, so it would otherwise never leave the uncovered set.
            m = uncov != w
            uncov, d_uncov, D = uncov[m], d_uncov[m], D[m]
            out.append((int(w), int(len(uncov))))
    return out

In [ ]:
import ast
import os


def completed_nodes(path):
    """Count the leading nodes already written to `path`, repairing a partial tail.

    An interrupted run can leave the last line truncated, so each line is parsed
    and checked against the node index it should carry. The file is truncated at
    the last line that passes, making the return value the next node to compute.
    """
    if not os.path.exists(path):
        return 0

    good_bytes = 0
    done = 0
    with open(path, "r+") as f:
        for line in f:
            stripped = line.strip()
            if not stripped or not line.endswith("\n"):
                break                              # blank or truncated tail
            try:
                index, payload = stripped.split(" ", 1)
                if int(index) != done:
                    break                          # out of order, stop here
                ast.literal_eval(payload)
            except (ValueError, SyntaxError):
                break
            good_bytes += len(line.encode())
            done += 1
        f.truncate(good_bytes)                     # drop the unusable tail
    return done


In [74]:
def write_adj_list(graph_path, vectors_path, out_path, alpha=1.0, limit=None,
                   chunk=100, report_every=1000, dtype="float64", resume=True):
    """Recompute coverage for every node and write the adj-list file.

    With `resume=True` an existing `out_path` is continued rather than rewritten:
    the nodes already in the file are kept and the loop picks up at the first one
    missing. Each line is flushed as it is written so an interrupt costs at most
    the node in flight.
    """
    if alpha < 1.0:
        raise ValueError(f"alpha must be >= 1, got {alpha}")
    alpha_sq = alpha ** 2

    graph, _ = read_csr(graph_path)
    total = min(len(graph), limit) if limit else len(graph)

    start_at = completed_nodes(out_path) if resume else 0
    if start_at >= total:
        print(f"{out_path} already has {start_at} nodes, nothing to do")
        return out_path
    if start_at:
        print(f"resuming {out_path} at node {start_at}/{total}", flush=True)

    V = xp.asarray(read_fbin(vectors_path), dtype=dtype)
    norms = xp.einsum("ij,ij->i", V, V)

    mode = "a" if start_at else "w"
    with open(out_path, mode) as out:
        # tqdm starts at the resume point so the bar reflects work actually left
        for i in tqdm(range(start_at, total), initial=start_at, total=total):
            nbrs = neighborhood_with_uncov(i, graph[i], V, norms, alpha_sq, chunk)
            out.write(f"{i} {nbrs}\n")
            out.flush()                    # a kill after this point loses nothing

    print(f"wrote {out_path} ({total} nodes)")
    return out_path


In [75]:
write_adj_list(GRAPH_PATH, BASE_FBIN, ADJ_PATH,
               alpha=COVERAGE_ALPHA, limit=LIMIT, dtype=DTYPE)

Exception ignored in: <generator object tqdm.__iter__ at 0x14685d8aeb00>
Traceback (most recent call last):
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/std.py", line 1196, in __iter__
    self.close()
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/std.py", line 1302, in close
    self.display(pos=0)
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/std.py", line 1495, in display
    self.sp(self.__str__() if msg is None else msg)
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/std.py", line 459, in print_status
    fp_write('\r' + s + (' ' * max(last_len[0] - len_s, 0)))
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/std.py", line 453, in fp_write
    fp_flush()
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/utils.py", line 196, in inner
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/ext3/miniforge3/envs/big_ann

KeyboardInterrupt: 

## 4. Modified Vamana (gamma stopping rule)

`code/vamana/` is this project's copy of Vamana, where `robustPrune` can stop on
achieved coverage instead of a fixed degree budget. With `-gamma g` it keeps adding
edges until they alpha-cover a `g` fraction of a uniform random sample of the
dataset, drawn once per build; `-S` sets the sample size and defaults to
`ceil(10 ln n)`.

`-R` still bounds the degree, because the graph allocates `n * (R + 1)` slots and
`update_neighbors` aborts past that. Selection stops at whichever comes first:
gamma reached, candidates exhausted, or `R` edges.

Build the binary first with `make` in `code/vamana/`.

In [25]:
mod_graph, MOD_BUILD_SECONDS = run_vamana(
    MOD_VAMANA_BIN, BASE_FBIN, MOD_GRAPH_PATH, R, L, ALPHA,
    gamma=GAMMA, sample_size=SAMPLE, extra_args=["-verbose"])
print(f"\nbuild time: {MOD_BUILD_SECONDS:.3f}s")


/scratch/pa2439/ANN-Search/navigable_graph_results/code/vamana/neighbors -R 32 -L 64 -alpha 1.0 -data_type float -dist_func Euclidian -base_path /scratch/pa2439/ANN-Search/datasets/coco_i2i-512-angular/base.fbin -graph_outfile /scratch/pa2439/ANN-Search/navigable_graph_results/new_results/modvamana-coco_i2i-512-euclidean-R32-alpha1-gamma0.9 -gamma 0.9 -verbose 

Data: detected 113287 points with dimension 512
Building graph...
gamma stopping rule: gamma = 0.9, |sample| = 117 (need 106 covered)
number of passes = 1
Pass 10% complete
Pass 20% complete
Pass 30% complete
Pass 40% complete
Pass 50% complete
Pass 60% complete
Pass 70% complete
Pass 80% complete
Pass 90% complete
Pass 100% complete
beam search time: total: 8.8222
bidirect time: total: 0.1361
prune time: total: 1.2976
Average visited: 79, Tail visited: 123
Vamana graph built with 113287 points and parameters R = 32, L = 64
Graph has average degree 7.622 and maximum degree 32
Graph built in 12.04 seconds
Parlay time: 12.2397
Wr

Coverage for the modified graph, written to its own adj-list. Same resumable
writer as section 3, so an interrupted run picks up where it stopped.

In [26]:
write_adj_list(MOD_GRAPH_PATH, BASE_FBIN, MOD_ADJ_PATH,
               alpha=COVERAGE_ALPHA, limit=LIMIT, dtype=DTYPE)


  1%|          | 735/113287 [00:06<15:33, 120.60it/s]
Exception ignored in: <generator object tqdm.__iter__ at 0x1508f10a8ee0>
Traceback (most recent call last):
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/std.py", line 1196, in __iter__
    self.close()
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/std.py", line 1303, in close
    fp_write('\n')
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/std.py", line 1287, in fp_write
    self.fp.write(str(s))
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/tqdm/utils.py", line 196, in inner
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3099, in write
    result = original_write(data, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/ext3/miniforge3/envs/big_ann/lib/python3.12/site-packages/ipykerne

KeyboardInterrupt: 

## 5. Compare regular vs modified Vamana

Both adj-lists record, for every edge, the number of points still uncovered after
that edge takes effect. The last entry of a neighbourhood is therefore what the
node achieves using all of its edges, and `coverage = 1 - uncov_last / n`.

- **min / max coverage** are over nodes: the worst and best any single node reaches.
  Min coverage is the gamma that `coverage_to_degree_analysis.py` would cap its
  sweep at.
- **out-degree** stats come from the neighbourhood lengths.
- **compute time** is the binary's own build timer, which excludes reading the
  fbin and writing the graph.

In [27]:
def graph_stats(adj_path, n_nodes, build_seconds=None):
    """Coverage and degree statistics for one adj-list file."""
    coverages, degrees, total_edges = [], [], 0
    with open(adj_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            payload = line if line.startswith("[") else line[line.index(" ") + 1:]
            nbrs = ast.literal_eval(payload)
            degrees.append(len(nbrs))
            total_edges += len(nbrs)
            # uncov after the last edge = what the node achieves using every edge
            last_uncov = nbrs[-1][1] if nbrs else n_nodes
            coverages.append(1.0 - last_uncov / n_nodes)

    if not degrees:
        raise ValueError(f"{adj_path} has no neighbourhoods")

    return {
        "nodes":            len(degrees),
        "min coverage":     100.0 * min(coverages),
        "max coverage":     100.0 * max(coverages),
        "avg out-degree":   float(np.mean(degrees)),
        "median out-degree": float(np.median(degrees)),
        "max out-degree":   int(max(degrees)),
        "total edges":      total_edges,
        "compute time (s)": build_seconds,
    }


In [28]:
import ast

n_nodes = len(read_fbin(BASE_FBIN))

stats = {
    "regular":  graph_stats(ADJ_PATH, n_nodes, globals().get("VAMANA_BUILD_SECONDS")),
    f"gamma={GAMMA:g}": graph_stats(MOD_ADJ_PATH, n_nodes, globals().get("MOD_BUILD_SECONDS")),
}

rows = ["min coverage", "max coverage", "avg out-degree", "median out-degree",
        "max out-degree", "total edges", "compute time (s)"]

def fmt(key, value):
    if value is None:
        return "n/a"                      # build ran in an earlier session
    if "coverage" in key:
        return f"{value:.2f}%"
    if key == "total edges":
        return f"{value:,}"
    if key in ("max out-degree",):
        return f"{value:d}"
    return f"{value:.3f}"

names = list(stats)
width = max(len(r) for r in rows) + 2
header = "metric".ljust(width) + "".join(n.rjust(16) for n in names)
print(header)
print("-" * len(header))
for r in rows:
    print(r.ljust(width) + "".join(fmt(r, stats[n][r]).rjust(16) for n in names))

print()
reg, mod = stats[names[0]], stats[names[1]]
print(f"nodes: {reg['nodes']:,} (regular), {mod['nodes']:,} (modified)")
if reg["nodes"] != mod["nodes"]:
    print("WARNING: different node counts - one adj-list is incomplete, "
          "so the comparison is not like-for-like")
print(f"edges:  {mod['total edges'] / reg['total edges']:.3f}x regular")
if reg["compute time (s)"] and mod["compute time (s)"]:
    print(f"build:  {mod['compute time (s)'] / reg['compute time (s)']:.3f}x regular")


metric                      regular       gamma=0.9
---------------------------------------------------
min coverage                  0.00%          45.60%
max coverage                100.00%          99.99%
avg out-degree               14.750           7.800
median out-degree            14.000           6.000
max out-degree                   32              32
total edges               1,671,039           5,733
compute time (s)             14.870          12.040

nodes: 113,287 (regular), 735 (modified)
edges:  0.003x regular
build:  0.810x regular


## Check the output

Parse the result the way the analysis scripts do and confirm `uncov` is non-increasing.

In [61]:
import ast

rows = []
with open(ADJ_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        source = line if line.startswith("[") else line[line.index(" ") + 1:]
        rows.append(ast.literal_eval(source))

n_nodes = len(read_fbin(BASE_FBIN))
for i, nb in enumerate(rows):
    uncovs = [u for _, u in nb]
    assert all(a >= b for a, b in zip(uncovs, uncovs[1:])), f"node {i} not monotone"
    assert all(0 <= n_nodes - u <= n_nodes for u in uncovs), f"node {i} out of range"

degrees = [len(nb) for nb in rows]
print(f"parsed {len(rows)} neighborhoods; uncov non-increasing and in range")
print(f"degree: mean {np.mean(degrees):.1f}, max {max(degrees)}")
print("cov_by_edge (node 0, first 8):", [n_nodes - u for _, u in rows[0]][:8])

parsed 113287 neighborhoods; uncov non-increasing and in range
degree: mean 14.8, max 32
cov_by_edge (node 0, first 8): [97278, 109429, 111121, 111848, 112312, 112559, 112758, 112917]


In [50]:
BUILT_DIR

PosixPath('/scratch/pa2439/ANN-Search/navigable_graph_results/new_results')